In [1]:
import re

with open("../data/messy_sequences.fasta") as f:
    headers = [line.rstrip("\n") for line in f if line.startswith(">")]

print(f"{len(headers)} headers loaded")
headers

8 headers loaded


['>sample_001|organism=Homo_sapiens|gene=BRCA1|len=120',
 '>Sample002 organism:Homo sapiens gene:TP53',
 '>sample-003 | Homo_sapiens | EGFR | length=150bp',
 '>SAMPLE_004;species=H.sapiens;target=BRCA1',
 '>sample005 Homo sapiens; TP53; 130 bp',
 '>seq6|Hsapiens|EGFR',
 '>Sample_007 organism=Homo_sapiens|gene=BRCA1|note:re-sequenced',
 '>sample-8 species:Homo_sapiens gene:TP53 len:NA']

In [2]:
sample_id_pattern = r"(?:sample[_-]?|seq)(\d+)"
organism_pattern = r"(?:organism[:=]|species[:=]|\|\s*|\s+)(Homo_sapiens|Homo sapiens|H\.sapiens|Hsapiens)"
gene_pattern = r"(?:gene[:=]|target[:=]|\|\s*|;\s*)(BRCA1|TP53|EGFR)"

def extract(pattern, text, group=0, flags=re.IGNORECASE):
    m = re.search(pattern, text, flags)
    return m.group(group) if m else None

print(f"{'sample_id':<12} {'organism':<14} {'gene'}")
print("-" * 40)

for h in headers:
    sid = extract(sample_id_pattern, h, group=1)
    org = extract(organism_pattern, h, group=1)
    gene = extract(gene_pattern, h, group=1)
    print(f"{str(sid):<12} {str(org):<14} {gene}")

sample_id    organism       gene
----------------------------------------
001          Homo_sapiens   BRCA1
002          Homo sapiens   TP53
003          Homo_sapiens   EGFR
004          H.sapiens      BRCA1
005          Homo sapiens   TP53
6            Hsapiens       EGFR
007          Homo_sapiens   BRCA1
8            Homo_sapiens   TP53


In [4]:
with open("../data/messy_sequences.fasta") as f:
    records = f.read().split(">")[1:]

with open("../data/cleaned_sequences_regex.fasta", "w") as out:
    for h, record in zip(headers, records):
        sid = extract(sample_id_pattern, h, 1)
        org = extract(organism_pattern, h, 1)
        gene = extract(gene_pattern, h, 1)
        seq = "".join(record.splitlines()[1:])

        out.write(f">sample_{sid}|organism={org}|gene={gene}\n{seq}\n")